# 04 · Distancias topológicas por día para ID6 a ID10

Este notebook implementa una serie de tiempo topológica por niño para `ID6` a `ID10`.

Objetivo:

- construir un diagrama de persistencia por niño y por día
- comparar diagramas entre días usando una distancia de Wasserstein
- identificar transiciones entre días con cambio topológico fuerte
- resumir el promedio de distancias por niño y por transición temporal

Decisiones técnicas fijadas aquí:

- la unidad temporal es el `día` (`dia_num`)
- cada día usa **todas** las etapas afectivas disponibles (`inicio`, `p1_before`, `p1_during`, `p1_after`, `p2_before`, `p2_during`, `p2_after`, `final`)
- si hay varias filas que corresponden al mismo día numérico, se agregan en una sola nube diaria
- la distancia principal se define como `wasserstein_h0 + wasserstein_h1`
- el criterio descriptivo de cambio fuerte será `distancia_total > media + 1 desviación estándar` dentro de cada niño


## 1. Setup

El cargador soporta tanto el Excel como el CSV versionado del repo. El análisis se guarda en una carpeta nueva para no mezclarlo con el notebook anterior.


In [ ]:
from pathlib import Path
import math
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from persim import wasserstein
from ripser import ripser
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
DATA_CANDIDATES = [
    BASE_DIR / 'DFBabyFaceValidacion.xlsx',
    BASE_DIR / 'DataFrameBabyFaceValidacion.xlsx',
    BASE_DIR / 'DataFrameBabyFaceValidacion.csv',
]
OUTPUT_DIR = BASE_DIR / '04_distancias_topologicas_por_dia_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_IDS = ['ID6', 'ID7', 'ID8', 'ID9', 'ID10']
STAGES = [
    ('inicio', 'valence_inicio', 'arousal_inicio'),
    ('p1_before', 'valence_p1_before', 'arousal_p1_before'),
    ('p1_during', 'valence_p1_during', 'arousal_p1_during'),
    ('p1_after', 'valence_p1_after', 'arousal_p1_after'),
    ('p2_before', 'valence_p2_before', 'arousal_p2_before'),
    ('p2_during', 'valence_p2_during', 'arousal_p2_during'),
    ('p2_after', 'valence_p2_after', 'arousal_p2_after'),
    ('final', 'valence_final', 'arousal_final'),
]
STAGE_ORDER = {stage: idx for idx, (stage, _, _) in enumerate(STAGES)}
MISSING_TOKENS = {'', '-', 'NA', 'NA ', 'N/A', 'nan', 'NaN', 'FIT_FAILED', 'FIT FAILED', 'FIND_FAILED', 'S/I', None}
EPSILON = 1e-10
DIM_COLORS = {0: '#1f78b4', 1: '#e31a1c'}
MAX_DAILY_POINTS_FOR_PH = 250


## 2. Funciones auxiliares

Estas funciones:

- cargan y limpian el archivo fuente
- reconstruyen la serie afectiva completa en formato largo
- agregan puntos por niño y por día
- calculan diagramas de persistencia H0 y H1
- miden distancias entre diagramas
- resumen los cambios entre días consecutivos


In [ ]:
def id_sort_key(value):
    match = re.search(r'(\d+)', str(value))
    return (int(match.group(1)), str(value)) if match else (10**9, str(value))

def parse_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    try:
        return float(s.replace(',', '.'))
    except Exception:
        return np.nan

def parse_day_number(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in MISSING_TOKENS:
        return np.nan
    match = re.search(r'\d+(?:\.\d+)?', s)
    return float(match.group(0)) if match else np.nan

def resolve_data_path():
    for path in DATA_CANDIDATES:
        if path.exists():
            return path
    raise FileNotFoundError('No se encontró un archivo de datos compatible.')

def load_and_clean(path):
    if path.suffix.lower() == '.csv':
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    df = df.copy()
    df.columns = [str(c).replace('\ufeff', '').strip() for c in df.columns]
    df['id'] = df['id'].astype(str).str.strip()
    df['dia_original'] = df['dia'].astype(str).str.strip()
    df['dia_num'] = df['dia'].apply(parse_day_number)
    binary_cols = ['rechaza', 'llora', 'toca', 'prueba', 'consume']
    va_cols = [c for c in df.columns if c.startswith('valence') or c.startswith('arousal')]
    for col in binary_cols + va_cols:
        if col in df.columns:
            df[col] = df[col].apply(parse_numeric)
    return df

def build_long_series(df):
    rows = []
    for stage, vcol, acol in STAGES:
        tmp = df[['id', 'dia_original', 'dia_num', vcol, acol]].copy()
        tmp.columns = ['id', 'dia_original', 'dia_num', 'valence', 'arousal']
        tmp['stage'] = stage
        tmp['stage_order'] = STAGE_ORDER[stage]
        rows.append(tmp)
    long_df = pd.concat(rows, ignore_index=True)
    long_df = long_df.sort_values(['id', 'dia_num', 'stage_order'], kind='stable').reset_index(drop=True)
    long_df = long_df.dropna(subset=['dia_num', 'valence', 'arousal']).copy()
    scaler = StandardScaler()
    long_df[['valence_scaled', 'arousal_scaled']] = scaler.fit_transform(long_df[['valence', 'arousal']])
    return long_df

def build_daily_point_clouds(long_df, target_ids=TARGET_IDS):
    sub = long_df[long_df['id'].isin(target_ids)].copy()
    records = []
    for (id_value, day_num), group in sub.groupby(['id', 'dia_num'], sort=True):
        labels = sorted(group['dia_original'].dropna().astype(str).unique().tolist())
        points = group[['valence_scaled', 'arousal_scaled']].to_numpy(dtype=float)
        records.append(
            {
                'id': id_value,
                'dia_num': float(day_num),
                'dia_labels': ' / '.join(labels),
                'n_points': int(len(points)),
                'points': points,
            }
        )
    daily_df = pd.DataFrame(records).sort_values(['id', 'dia_num'], key=lambda s: s.map(id_sort_key) if s.name == 'id' else s).reset_index(drop=True)
    return daily_df

def compute_diagrams(points):
    if len(points) == 0:
        return [np.empty((0, 2)), np.empty((0, 2))]
    if len(points) == 1:
        return [np.array([[0.0, np.inf]], dtype=float), np.empty((0, 2))]
    kwargs = {'maxdim': 1}
    if len(points) > MAX_DAILY_POINTS_FOR_PH:
        kwargs['n_perm'] = MAX_DAILY_POINTS_FOR_PH
    dgms = ripser(points, **kwargs)['dgms']
    if len(dgms) == 1:
        dgms.append(np.empty((0, 2)))
    return [np.asarray(dgm, dtype=float) for dgm in dgms[:2]]

def finite_positive_diagram(diagram):
    if diagram.size == 0:
        return np.empty((0, 2))
    finite_mask = np.isfinite(diagram[:, 0]) & np.isfinite(diagram[:, 1])
    filtered = diagram[finite_mask]
    if filtered.size == 0:
        return np.empty((0, 2))
    persistence = filtered[:, 1] - filtered[:, 0]
    return filtered[persistence > EPSILON]

def wasserstein_to_empty(diagram):
    if diagram.size == 0:
        return 0.0
    persistence = diagram[:, 1] - diagram[:, 0]
    return float(np.sqrt(np.sum((persistence / np.sqrt(2.0)) ** 2)))

def diagram_wasserstein(diag_a, diag_b):
    if len(diag_a) == 0 and len(diag_b) == 0:
        return 0.0
    if len(diag_a) == 0:
        return wasserstein_to_empty(diag_b)
    if len(diag_b) == 0:
        return wasserstein_to_empty(diag_a)
    return float(wasserstein(diag_a, diag_b))

def combined_distance(diagrams_a, diagrams_b):
    d_h0 = diagram_wasserstein(diagrams_a[0], diagrams_b[0])
    d_h1 = diagram_wasserstein(diagrams_a[1], diagrams_b[1])
    return d_h0, d_h1, d_h0 + d_h1

def ensure_square_axes(ax, max_value):
    limit = max(max_value, 1e-3) * 1.05
    ax.plot([0, limit], [0, limit], linestyle='--', color='gray', linewidth=1)
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)

def plot_daily_diagram(ax, diagrams, title, max_value):
    has_points = False
    for dim in (0, 1):
        diagram = diagrams[dim]
        if diagram.size == 0:
            continue
        has_points = True
        ax.scatter(diagram[:, 0], diagram[:, 1], s=18, alpha=0.7, color=DIM_COLORS[dim], label=f'H{dim}')
    ensure_square_axes(ax, max_value)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')
    ax.grid(alpha=0.2)
    if not has_points:
        ax.text(0.5, 0.5, 'Sin features finitas', ha='center', va='center', transform=ax.transAxes)

def make_distance_matrix(day_rows):
    n = len(day_rows)
    matrix_h0 = np.zeros((n, n), dtype=float)
    matrix_h1 = np.zeros((n, n), dtype=float)
    matrix_total = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(i + 1, n):
            d_h0, d_h1, d_total = combined_distance(day_rows[i]['diagrams'], day_rows[j]['diagrams'])
            matrix_h0[i, j] = matrix_h0[j, i] = d_h0
            matrix_h1[i, j] = matrix_h1[j, i] = d_h1
            matrix_total[i, j] = matrix_total[j, i] = d_total
    return matrix_h0, matrix_h1, matrix_total

def summarize_consecutive_distances(day_rows):
    rows = []
    for prev_row, next_row in zip(day_rows[:-1], day_rows[1:]):
        d_h0, d_h1, d_total = combined_distance(prev_row['diagrams'], next_row['diagrams'])
        rows.append(
            {
                'id': prev_row['id'],
                'dia_from': prev_row['dia_num'],
                'dia_to': next_row['dia_num'],
                'transition': f"{prev_row['dia_num']:g}->{next_row['dia_num']:g}",
                'n_points_from': prev_row['n_points'],
                'n_points_to': next_row['n_points'],
                'wasserstein_h0': d_h0,
                'wasserstein_h1': d_h1,
                'distance_total': d_total,
            }
        )
    if not rows:
        return pd.DataFrame(columns=['id', 'dia_from', 'dia_to', 'transition', 'n_points_from', 'n_points_to', 'wasserstein_h0', 'wasserstein_h1', 'distance_total'])
    df = pd.DataFrame(rows)
    for id_value, sub in df.groupby('id'):
        mean_value = float(sub['distance_total'].mean())
        std_value = float(sub['distance_total'].std(ddof=0))
        threshold = mean_value + std_value
        mask = df['id'] == id_value
        df.loc[mask, 'mean_distance_total_child'] = mean_value
        df.loc[mask, 'std_distance_total_child'] = std_value
        df.loc[mask, 'threshold_change_total'] = threshold
        df.loc[mask, 'strong_change_total'] = df.loc[mask, 'distance_total'] > threshold
    return df


## 3. Carga de datos y construcción de nubes diarias

Aquí reorganizamos la información a nivel `(niño, día)`. Si un mismo día aparece en varias filas con etiquetas como `E`, `P` o variantes, todas esas observaciones se agregan al mismo punto temporal `dia_num`.


In [ ]:
data_path = resolve_data_path()
df = load_and_clean(data_path)
long_series = build_long_series(df)
daily_df = build_daily_point_clouds(long_series)

print(f'Archivo cargado: {data_path.name}')
print('\nDías disponibles por niño:')
for id_value, sub in daily_df.groupby('id'):
    days = ', '.join(f"{day:g}" for day in sub['dia_num'].tolist())
    print(f'- {id_value}: {len(sub)} días -> {days}')

daily_summary = daily_df[['id', 'dia_num', 'dia_labels', 'n_points']].copy()
daily_summary.to_csv(OUTPUT_DIR / 'daily_point_clouds_id6_id10.csv', index=False)
print('\nResumen diario guardado en daily_point_clouds_id6_id10.csv')
print('\nPrimeras filas del resumen diario:')
print(daily_summary.head(20).to_string(index=False))


## 4. Diagramas de persistencia por niño y por día

Cada figura guarda los diagramas diarios de un niño. Esto permite inspeccionar visualmente cómo cambia la estructura topológica antes de pasar a las distancias entre días.


In [ ]:
daily_records_by_id = {}
for id_value, sub in daily_df.groupby('id'):
    rows = []
    for row in sub.sort_values('dia_num').to_dict('records'):
        diagrams = [finite_positive_diagram(dgm) for dgm in compute_diagrams(row['points'])]
        row['diagrams'] = diagrams
        rows.append(row)
    daily_records_by_id[id_value] = rows

for id_value, rows in daily_records_by_id.items():
    all_deaths = []
    for row in rows:
        for dim in (0, 1):
            diagram = row['diagrams'][dim]
            if diagram.size:
                all_deaths.append(diagram[:, 1])
    max_value = float(np.max(np.concatenate(all_deaths))) if all_deaths else 1.0
    ncols = 4
    nrows = math.ceil(len(rows) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    for ax, row in zip(axes.flat, rows):
        ax.set_visible(True)
        plot_daily_diagram(ax, row['diagrams'], title=f"día {row['dia_num']:g}\n{row['dia_labels']}", max_value=max_value)
        ax.legend(loc='lower right', fontsize=7)
    fig.suptitle(f'Diagramas de persistencia por día · {id_value}', fontsize=16)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'diagramas_por_dia_{id_value}.png', dpi=180, bbox_inches='tight')
    plt.show()


## 5. Distancias entre diagramas por día

La señal principal del notebook es la distancia entre diagramas de días consecutivos. Aquí usamos Wasserstein en `H0` y `H1`, y definimos una distancia total como la suma de ambas.

Además, generamos:

- una matriz completa día vs. día por niño
- una serie temporal de distancias consecutivas por niño
- un criterio descriptivo de cambio fuerte: `media + 1 sd`


In [ ]:
consecutive_frames = []
for id_value, rows in daily_records_by_id.items():
    matrix_h0, matrix_h1, matrix_total = make_distance_matrix(rows)
    labels = [f"{row['dia_num']:g}" for row in rows]
    pd.DataFrame(matrix_total, index=labels, columns=labels).to_csv(OUTPUT_DIR / f'distance_matrix_total_{id_value}.csv', index_label='dia')
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    for ax, matrix, title in zip(
        axes,
        [matrix_h0, matrix_h1, matrix_total],
        ['Wasserstein H0', 'Wasserstein H1', 'Distancia total'],
    ):
        image = ax.imshow(matrix, cmap='viridis')
        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=90)
        ax.set_yticklabels(labels)
        ax.set_title(f'{id_value} · {title}')
        ax.set_xlabel('día')
        ax.set_ylabel('día')
        fig.colorbar(image, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'heatmap_distancias_{id_value}.png', dpi=180, bbox_inches='tight')
    plt.show()

    consecutive_frames.append(summarize_consecutive_distances(rows))

distances_df = pd.concat(consecutive_frames, ignore_index=True)
distances_df.to_csv(OUTPUT_DIR / 'distancias_consecutivas_id6_id10.csv', index=False)

mean_by_child = (
    distances_df.groupby('id')
    .agg(
        n_transitions=('transition', 'count'),
        mean_wasserstein_h0=('wasserstein_h0', 'mean'),
        mean_wasserstein_h1=('wasserstein_h1', 'mean'),
        mean_distance_total=('distance_total', 'mean'),
        std_distance_total=('distance_total', 'std'),
        max_distance_total=('distance_total', 'max'),
    )
    .reset_index()
    .sort_values('id', key=lambda s: s.map(id_sort_key))
)
mean_by_child.to_csv(OUTPUT_DIR / 'promedio_distancias_por_nino.csv', index=False)

mean_by_transition = (
    distances_df.groupby(['dia_from', 'dia_to', 'transition'])
    .agg(
        n_ninos=('id', 'count'),
        mean_distance_total=('distance_total', 'mean'),
        mean_wasserstein_h0=('wasserstein_h0', 'mean'),
        mean_wasserstein_h1=('wasserstein_h1', 'mean'),
    )
    .reset_index()
    .sort_values(['dia_from', 'dia_to'])
)
mean_by_transition.to_csv(OUTPUT_DIR / 'promedio_distancias_por_transicion.csv', index=False)

strong_changes = distances_df[distances_df['strong_change_total']].copy()
strong_changes.to_csv(OUTPUT_DIR / 'cambios_fuertes_descriptivos.csv', index=False)

print('Resumen del promedio de distancias por niño:')
print(mean_by_child.to_string(index=False))
print('\nPromedio de distancias por transición temporal:')
print(mean_by_transition.to_string(index=False))
print('\nCambios fuertes descriptivos detectados:')
if len(strong_changes):
    print(strong_changes[['id', 'transition', 'distance_total', 'threshold_change_total']].to_string(index=False))
else:
    print('No se detectaron transiciones por encima del umbral media + 1 sd.')


In [ ]:
fig, axes = plt.subplots(len(TARGET_IDS), 1, figsize=(12, 3.2 * len(TARGET_IDS)), sharex=False)
if len(TARGET_IDS) == 1:
    axes = [axes]

for ax, id_value in zip(axes, TARGET_IDS):
    sub = distances_df[distances_df['id'] == id_value].copy()
    x = np.arange(len(sub))
    ax.plot(x, sub['distance_total'], marker='o', linewidth=2, color='#0f766e', label='distancia total')
    ax.plot(x, sub['wasserstein_h0'], marker='o', linewidth=1.5, color=DIM_COLORS[0], alpha=0.85, label='H0')
    ax.plot(x, sub['wasserstein_h1'], marker='o', linewidth=1.5, color=DIM_COLORS[1], alpha=0.85, label='H1')
    if len(sub):
        threshold = float(sub['threshold_change_total'].iloc[0])
        ax.axhline(threshold, linestyle='--', color='#b45309', linewidth=1.5, label='media + 1 sd')
        strong_sub = sub[sub['strong_change_total']]
        if len(strong_sub):
            strong_idx = strong_sub.index.map(lambda idx: sub.index.get_loc(idx)).to_numpy()
            ax.scatter(strong_idx, strong_sub['distance_total'], color='#b91c1c', s=55, zorder=5, label='cambio fuerte')
    ax.set_xticks(x)
    ax.set_xticklabels(sub['transition'].tolist(), rotation=45, ha='right')
    ax.set_title(f'{id_value} · distancias entre días consecutivos')
    ax.set_ylabel('distancia')
    ax.grid(alpha=0.25)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('transición temporal')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'serie_distancias_consecutivas_id6_id10.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 4.8))
x = np.arange(len(mean_by_transition))
ax.bar(x, mean_by_transition['mean_distance_total'], color='#155e75')
ax.set_xticks(x)
ax.set_xticklabels(mean_by_transition['transition'].tolist(), rotation=45, ha='right')
ax.set_title('Promedio de distancia total por transición temporal')
ax.set_ylabel('promedio de distancia')
ax.set_xlabel('transición')
ax.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'promedio_distancias_por_transicion.png', dpi=180, bbox_inches='tight')
plt.show()


## 6. Salidas guardadas

Este notebook guarda en `04_distancias_topologicas_por_dia_outputs/`:

- `daily_point_clouds_id6_id10.csv`
- `diagramas_por_dia_ID6.png` ... `diagramas_por_dia_ID10.png`
- `heatmap_distancias_ID6.png` ... `heatmap_distancias_ID10.png`
- `distancias_consecutivas_id6_id10.csv`
- `promedio_distancias_por_nino.csv`
- `promedio_distancias_por_transicion.csv`
- `cambios_fuertes_descriptivos.csv`
- `serie_distancias_consecutivas_id6_id10.png`
- `promedio_distancias_por_transicion.png`

Interpretación principal:

- una distancia pequeña implica días topológicamente parecidos
- una distancia grande implica un cambio fuerte entre días
- el foco no es “qué día es raro” sino “entre qué días ocurre el cambio”
